# PLS — estudo conceitual

Random Forest e PLS desta base usam os mesmos 17 preditores. Vários deles são defasagens e médias do próprio PM2.5, então andam juntos. Uma regressão comum reparte mal esse crédito: coeficientes vizinhos ficam instáveis e podem sair com sinais opostos.

O PLS monta poucos componentes. Cada componente é uma combinação dos preditores escolhida para acompanhar o PM2.5 da hora seguinte. A regressão é feita nesses componentes, não nas 17 colunas cruas.

| | OLS | componente principal | PLS |
|---|---|---|---|
| Colinearidade | coeficientes instáveis | reduz a dimensão de X | reduz a dimensão de X |
| Olha o alvo ao construir o eixo? | — | não | sim |
| O que a grade escolhe | — | variância de X | covariância entre X e y |

A matriz exploratória de 671 colunas continua em `01_dados`. Ela não é a matriz deste ajuste.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

current = Path.cwd()
while current != current.parent and not (current / "bases" / "grupo3").exists():
    current = current.parent
ROOT = current
BASE = ROOT / "analyses" / "grupo3"
OUT = BASE / "03_resultados" / "outputs"
print(ROOT)

frame = pd.read_csv(BASE / "01_dados" / "outputs" / "features_modelagem_base3.csv", parse_dates=["timestamp"])
features = [
    "pm25_t", "pm25_t_1", "pm25_t_2", "pm25_t_23", "pm25_roll_mean_3", "pm25_roll_mean_24",
    "local_PM10", "local_CO", "local_TEMP", "local_WSPM", "local_DEWP",
    "Changping_PM2.5", "Wanliu_PM2.5",
    "target_hour_sin", "target_hour_cos", "target_dow_sin", "target_dow_cos",
]
print(frame["split"].value_counts().to_string())
print("preditores", len(features), "linhas", len(frame))


/Users/leticia.nsilva/Desktop/personal/pls-regration-comparation
split
train_interno        19494
test_final            6023
validacao_interna     4545
preditores 17 linhas 30062


## Colinearidade do histórico de PM2.5

As seis colunas abaixo são funções do mesmo PM2.5. A correlação alta é esperada e é o motivo de não ler um coeficiente isolado como efeito separado.

In [2]:
history = ["pm25_t", "pm25_t_1", "pm25_t_2", "pm25_t_23", "pm25_roll_mean_3", "pm25_roll_mean_24"]
corr = frame.loc[frame["split"] == "train_interno", history].corr().round(3)
print(corr.to_string())


                   pm25_t  pm25_t_1  pm25_t_2  pm25_t_23  pm25_roll_mean_3  pm25_roll_mean_24
pm25_t              1.000     0.967     0.919      0.357             0.978              0.726
pm25_t_1            0.967     1.000     0.968      0.371             0.994              0.757
pm25_t_2            0.919     0.968     1.000      0.384             0.978              0.784
pm25_t_23           0.357     0.371     0.384      1.000             0.377              0.726
pm25_roll_mean_3    0.978     0.994     0.978      0.377             1.000              0.768
pm25_roll_mean_24   0.726     0.757     0.784      0.726             0.768              1.000


## Componentes principais e número de componentes do PLS

O componente principal resume X sem olhar y. O laço abaixo ajusta o PLS no treino interno e mede o R² na validação, com preditores padronizados. Esse R² é um retrato de um único ajuste. A escolha usada no teste é o MAE do walk-forward diário, gravado no notebook de implementação.

In [3]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

train = frame[frame["split"] == "train_interno"]
valid = frame[frame["split"] == "validacao_interna"]
scaler = StandardScaler().fit(train[features])
x_train = scaler.transform(train[features])
x_valid = scaler.transform(valid[features])
y_train = train["y"].to_numpy(float)
y_valid = valid["y"].to_numpy(float)

pca = PCA().fit(x_train)
print("variância acumulada dos componentes principais")
print(np.cumsum(pca.explained_variance_ratio_).round(3))

rows = []
for n_components in (1, 2, 4, 8, 12, 14, 16):
    model = PLSRegression(n_components=n_components, scale=True, max_iter=500)
    model.fit(x_train, y_train)
    rows.append({
        "n_components": n_components,
        "R2_treino": r2_score(y_train, model.predict(x_train)),
        "R2_validacao": r2_score(y_valid, model.predict(x_valid)),
    })
comparison = pd.DataFrame(rows)
comparison.to_csv(BASE / "04_especialista_PLS" / "outputs" / "r2_um_ajuste.csv", index=False)
print(comparison.round(4).to_string(index=False))


variância acumulada dos componentes principais
[0.455 0.569 0.655 0.719 0.779 0.837 0.893 0.929 0.949 0.967 0.979 0.987
 0.992 0.996 0.999 1.    1.   ]
 n_components  R2_treino  R2_validacao
            1     0.8851        0.9046
            2     0.9044        0.9214
            4     0.9251        0.9403
            8     0.9392        0.9537
           12     0.9411        0.9558
           14     0.9412        0.9557
           16     0.9413        0.9560


/Users/leticia.nsilva/Desktop/personal/pls-regration-comparation/.venv/lib/python3.13/site-packages/sklearn/cross_decomposition/_pls.py:531: RuntimeWarning: divide by zero encountered in matmul
  Ypred = X @ self.coef_.T + self.intercept_
/Users/leticia.nsilva/Desktop/personal/pls-regration-comparation/.venv/lib/python3.13/site-packages/sklearn/cross_decomposition/_pls.py:531: RuntimeWarning: overflow encountered in matmul
  Ypred = X @ self.coef_.T + self.intercept_
/Users/leticia.nsilva/Desktop/personal/pls-regration-comparation/.venv/lib/python3.13/site-packages/sklearn/cross_decomposition/_pls.py:531: RuntimeWarning: invalid value encountered in matmul
  Ypred = X @ self.coef_.T + self.intercept_
/Users/leticia.nsilva/Desktop/personal/pls-regration-comparation/.venv/lib/python3.13/site-packages/sklearn/cross_decomposition/_pls.py:531: RuntimeWarning: divide by zero encountered in matmul
  Ypred = X @ self.coef_.T + self.intercept_
/Users/leticia.nsilva/Desktop/personal/pls-regratio